# مشروع OCR لخط اليد — النسخة المدمجة للمراقبة والتدريب على Google Colab

النسخة دي مدمجة ومجهزة لكولاب، وبتجمع بين:
- استخراج النصوص من PDF أو الصور.
- دمج **TrOCR + EasyOCR + Tesseract**.
- مراجعة الكلمات والجمل يدويًا.
- بناء قاموس تصحيح تلقائي من تصحيحاتك السابقة.
- إنشاء **لوغات تشغيل ومراقبة** قابلة للمتابعة.
- تحليل اللوغات والـ feedback السابقة داخل نفس المشروع.
- تصدير بيانات التدريب وبدء **LoRA fine-tuning** عند توفر عدد كافٍ من العينات.

> الهدف من النسخة دي هو تقليل فقدان الشغل في Colab، وتحويل المشروع إلى pipeline متكاملة قابلة للاستمرار والتوسعة.


## ما الذي تم دمجه وتحسينه في النسخة دي

اعتمادًا على الملفات المرفقة سابقًا، تم تحسين المشروع في النقاط دي:

1. **دمج feedback السابق** لبناء correction dictionary تراكمي.
2. **قراءة ومعاينة اللوغات السابقة** مثل:
   - `app.log`
   - `processing_stats.json`
   - `user_corrections_feedback.csv`
3. **تقوية الاستمرارية** عبر:
   - checkpoint بعد كل صفحة
   - run history
   - JSONL event logs
   - health snapshots
4. **تحسينات المعالجة**:
   - deskew مبسط
   - preprocessing مرن
   - fallback segmentation
   - دعم PDF والصور
5. **تحسينات المتابعة**:
   - Monitoring Dashboard
   - نسخ احتياطي سريع
   - جداول SQLite إضافية للـ runs والمراجعات

### خلاصة مختصرة من الملفات السابقة
- آخر إحصائية مرفقة أشارت تقريبًا إلى: **3 صفحات / 403 كلمة**.
- `app.log` أظهر **إعادة تشغيل للـ kernel**، لذلك تم تقوية الحفظ المرحلي والـ monitoring.
- ملف `user_corrections_feedback` احتوى على عدد كبير من التصحيحات، لذلك تم تطوير آلية correction dictionary.


In [1]:
# ============================================================
# الخلية 1: تثبيت المكتبات المطلوبة
# ============================================================
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara
!pip -q install pdf2image easyocr pyspellchecker langdetect transformers peft     huggingface_hub datasets Pillow opencv-python-headless pandas numpy matplotlib     ipywidgets scikit-learn pytesseract arabic-reshaper python-bidi ar-corrector


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils tesseract-ocr-ara
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 831 kB of archives.
After this operation, 2,144 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-ara all 1:4.00~git30-7274cfa-1.1 [645 kB]
Fetched 831 kB in 1s (1,259 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package tesseract-ocr-ara.
Preparing to unpack ...

In [2]:
# ============================================================
# الخلية 2: الاستيرادات والإعداد العام
# ============================================================
from google.colab import drive, files, userdata, patches
import os, io, gc, cv2, json, time, shutil, sqlite3, logging, platform
from pathlib import Path
from dataclasses import dataclass
from logging.handlers import RotatingFileHandler
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import easyocr
import pytesseract
import torch
import ipywidgets as widgets

from PIL import Image
from pdf2image import convert_from_path
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from ar_corrector.corrector import Corrector
from IPython.display import display, clear_output

DetectorFactory.seed = 0

drive.mount('/content/drive')

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('✅ تم تحميل HF_TOKEN من Colab Secrets')
except Exception:
    HF_TOKEN = None
    print('ℹ️ لم يتم العثور على HF_TOKEN — عادي لو مش هترفع على Hugging Face دلوقتي.')


Mounted at /content/drive
✅ تم تحميل HF_TOKEN من Colab Secrets


In [6]:
# ============================================================
# الخلية 3: الإعدادات والمسارات
# ============================================================
@dataclass
class ProjectConfig:
    project_root: str = '/content/drive/MyDrive/Handwritten_OCR_Integrated'
    input_pdf_path: str = '/content/drive/MyDrive/python notes.pdf'
    model_name: str = 'David-Magdy/TR_OCR_LARGE'
    default_dpi: int = 300
    easyocr_languages: tuple = ('en', 'ar')
    use_gpu_if_available: bool = True
    min_box_width: int = 15
    min_box_height: int = 10
    line_y_tolerance: int = 25
    correction_min_frequency: int = 2
    review_queue_order: str = 'low_confidence_first'

CFG = ProjectConfig()
PROJECT_ROOT = Path(CFG.project_root)
DIRS = {
    'data': PROJECT_ROOT / 'data',
    'db': PROJECT_ROOT / 'database',
    'logs': PROJECT_ROOT / 'logs',
    'exports': PROJECT_ROOT / 'exports',
    'cache': PROJECT_ROOT / 'models_cache',
    'artifacts': PROJECT_ROOT / 'artifacts',
    'backups': PROJECT_ROOT / 'backups',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DIRS['db'] / 'handwriting_data.db'
CHECKPOINT_FILE = DIRS['artifacts'] / 'ocr_checkpoint.json'
CORRECTION_DICT_PATH = DIRS['artifacts'] / 'correction_dict.json'
FEEDBACK_CSV = DIRS['logs'] / 'user_corrections_feedback.csv'
PROCESSING_STATS_JSON = DIRS['logs'] / 'processing_stats.json'
RUNS_CSV = DIRS['logs'] / 'runs_history.csv'
HEALTH_JSON = DIRS['logs'] / 'system_health.json'
APP_LOG = DIRS['logs'] / 'ocr_runtime.log'
EVENTS_JSONL = DIRS['logs'] / 'ocr_events.jsonl'
NOTEBOOK_BOOTSTRAP_SUMMARY = DIRS['artifacts'] / 'attached_logs_bootstrap_summary.json'

os.environ['TRANSFORMERS_CACHE'] = str(DIRS['cache'])
os.environ['TORCH_HOME'] = str(DIRS['cache'])

if not FEEDBACK_CSV.exists():
    pd.DataFrame(columns=['timestamp','image_id','original_text','corrected_text','status']).to_csv(FEEDBACK_CSV, index=False, encoding='utf-8-sig')
if not RUNS_CSV.exists():
    pd.DataFrame(columns=['run_id','timestamp','input_path','pages_processed','words_processed','verified_count','avg_confidence','status','duration_sec']).to_csv(RUNS_CSV, index=False, encoding='utf-8-sig')

BOOTSTRAP_SUMMARY = {
    'created_from_attached_files': True,
    'observed_processing_stats': {'pages': 3, 'words': 403},
    'observed_app_log_issue': 'Kernel restarts appeared in previous session logs, so checkpointing and monitoring were strengthened.',
    'observed_feedback_signal': 'Large correction history detected; correction dictionary and feedback ingestion were improved.'
}
with open(NOTEBOOK_BOOTSTRAP_SUMMARY, 'w', encoding='utf-8') as f:
    json.dump(BOOTSTRAP_SUMMARY, f, ensure_ascii=False, indent=2)

print('✅ تم تجهيز مسارات المشروع')
print('📁 PROJECT_ROOT =', PROJECT_ROOT)

✅ تم تجهيز مسارات المشروع
📁 PROJECT_ROOT = /content/drive/MyDrive/Handwritten_OCR_Integrated


In [7]:
# ============================================================
# الخلية 4: نظام اللوغات والمراقبة
# ============================================================
LOGGER = logging.getLogger('handwriting_ocr')
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()

formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
file_handler = RotatingFileHandler(APP_LOG, maxBytes=2_000_000, backupCount=5, encoding='utf-8')
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
LOGGER.addHandler(file_handler)
LOGGER.addHandler(stream_handler)


def log_event(event_type, payload=None, level='info'):
    payload = payload or {}
    record = {
        'timestamp': datetime.now().isoformat(),
        'event_type': event_type,
        'payload': payload,
    }
    with open(EVENTS_JSONL, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '
')
    msg = f"{event_type} | {json.dumps(payload, ensure_ascii=False)}"
    getattr(LOGGER, level.lower(), LOGGER.info)(msg)


def write_health_snapshot(extra=None):
    extra = extra or {}
    try:
        snapshot = {
            'timestamp': datetime.now().isoformat(),
            'python': platform.python_version(),
            'platform': platform.platform(),
            'torch_cuda_available': bool(torch.cuda.is_available()),
            'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
            'cwd': os.getcwd(),
            'extra': extra,
        }
    except Exception as e:
        snapshot = {
            'timestamp': datetime.now().isoformat(),
            'error': str(e),
            'extra': extra,
        }
    with open(HEALTH_JSON, 'w', encoding='utf-8') as f:
        json.dump(snapshot, f, ensure_ascii=False, indent=2)
    return snapshot

log_event('project_started', {'project_root': str(PROJECT_ROOT)})
write_health_snapshot({'stage': 'startup'})
print('✅ نظام اللوغات جاهز')

SyntaxError: unterminated string literal (detected at line 25) (995391271.py, line 25)

In [ ]:
# ============================================================
# الخلية 5: استيراد اللوغات/الـ feedback السابقة إن وجدت
# ============================================================
def import_uploaded_artifacts():
    # ارفع ملفات مثل app.log / processing_stats.json / user_corrections_feedback.csv من جهازك
    uploaded = files.upload()
    imported = []
    for fname, content in uploaded.items():
        target = DIRS['artifacts'] / fname
        with open(target, 'wb') as f:
            f.write(content)
        imported.append(str(target))
    log_event('artifacts_uploaded', {'count': len(imported), 'files': imported})
    print('✅ تم استيراد الملفات:')
    for p in imported:
        print(' -', p)
    return imported


def merge_feedback_csv(external_csv_path):
    external_csv_path = Path(external_csv_path)
    if not external_csv_path.exists():
        raise FileNotFoundError(external_csv_path)
    current = pd.read_csv(FEEDBACK_CSV, encoding='utf-8-sig') if FEEDBACK_CSV.exists() else pd.DataFrame()
    incoming = pd.read_csv(external_csv_path)
    merged = pd.concat([current, incoming], ignore_index=True)
    merged = merged.drop_duplicates(subset=['timestamp','image_id','original_text','corrected_text','status'], keep='last')
    merged.to_csv(FEEDBACK_CSV, index=False, encoding='utf-8-sig')
    log_event('feedback_merged', {'source': str(external_csv_path), 'rows_after_merge': int(len(merged))})
    print(f'✅ تم دمج feedback. عدد الصفوف الحالي: {len(merged)}')
    return merged


def preview_external_logs(log_path=None, stats_path=None, feedback_path=None):
    if log_path and Path(log_path).exists():
        print('
📄 معاينة app.log')
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            for i, line in enumerate(f):
                if i >= 15:
                    break
                print(line.rstrip())
    if stats_path and Path(stats_path).exists():
        print('
📊 معاينة processing_stats.json')
        with open(stats_path, 'r', encoding='utf-8') as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
    if feedback_path and Path(feedback_path).exists():
        print('
📝 معاينة feedback CSV')
        display(pd.read_csv(feedback_path).head(10))

# أمثلة:
# import_uploaded_artifacts()
# merge_feedback_csv('/content/user_corrections_feedback.csv')
# preview_external_logs('/content/app.log', '/content/processing_stats.json', '/content/user_corrections_feedback.csv')


In [ ]:
# ============================================================
# الخلية 6: قاعدة البيانات والـ checkpoint
# ============================================================
def get_conn():
    return sqlite3.connect(DB_PATH)


def init_db():
    with get_conn() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS handwriting_data (
                image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                image_data BLOB,
                predicted_text TEXT,
                raw_text TEXT,
                status TEXT,
                confidence REAL,
                model_source TEXT,
                page_num INTEGER,
                x INTEGER, y INTEGER, w INTEGER, h INTEGER,
                created_at TEXT,
                updated_at TEXT,
                run_id TEXT
            )
        ''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS processing_runs (
                run_id TEXT PRIMARY KEY,
                started_at TEXT,
                ended_at TEXT,
                input_path TEXT,
                start_page INTEGER,
                end_page INTEGER,
                dpi INTEGER,
                words_processed INTEGER,
                avg_confidence REAL,
                status TEXT,
                notes TEXT
            )
        ''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS review_events (
                review_id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT,
                image_id INTEGER,
                original_text TEXT,
                corrected_text TEXT,
                action TEXT,
                reviewer TEXT
            )
        ''')
        conn.execute('CREATE INDEX IF NOT EXISTS idx_handwriting_status ON handwriting_data(status)')
        conn.execute('CREATE INDEX IF NOT EXISTS idx_handwriting_page ON handwriting_data(page_num)')
        conn.execute('CREATE INDEX IF NOT EXISTS idx_handwriting_run ON handwriting_data(run_id)')
    log_event('db_initialized', {'db_path': str(DB_PATH)})


def save_checkpoint(data):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None


def clear_checkpoint():
    if CHECKPOINT_FILE.exists():
        CHECKPOINT_FILE.unlink()

init_db()
print('✅ تم تهيئة قاعدة البيانات والـ checkpoint')


In [ ]:
# ============================================================
# الخلية 7: تحميل النماذج والمدققات
# ============================================================
DEVICE = torch.device('cuda' if (torch.cuda.is_available() and CFG.use_gpu_if_available) else 'cpu')
LOGGER.info(f'Using device: {DEVICE}')

print('⏳ جاري تحميل TrOCR / EasyOCR / spell checkers ...')
load_start = time.time()

trocr_processor = TrOCRProcessor.from_pretrained(CFG.model_name, cache_dir=str(DIRS['cache']))
trocr_model = VisionEncoderDecoderModel.from_pretrained(CFG.model_name, cache_dir=str(DIRS['cache'])).to(DEVICE)

lora_save_path = DIRS['cache'] / 'trocr_lora_finetuned'
if lora_save_path.exists():
    try:
        from peft import PeftModel
        trocr_model = PeftModel.from_pretrained(trocr_model, str(lora_save_path)).to(DEVICE)
        print('✅ تم تحميل LoRA fine-tuned weights')
    except Exception as e:
        print('⚠️ تعذر تحميل LoRA، سيتم استخدام النموذج الأساسي:', e)

EASYOCR_DRIVE_PATH = PROJECT_ROOT / '.EasyOCR'
LOCAL_EASYOCR_PATH = Path.home() / '.EasyOCR'
if not EASYOCR_DRIVE_PATH.exists() and LOCAL_EASYOCR_PATH.exists():
    shutil.move(str(LOCAL_EASYOCR_PATH), str(EASYOCR_DRIVE_PATH))
if not LOCAL_EASYOCR_PATH.exists() and EASYOCR_DRIVE_PATH.exists():
    try:
        os.symlink(EASYOCR_DRIVE_PATH, LOCAL_EASYOCR_PATH)
    except Exception:
        pass
if not EASYOCR_DRIVE_PATH.exists():
    EASYOCR_DRIVE_PATH.mkdir(parents=True, exist_ok=True)

try:
    easy_reader = easyocr.Reader(list(CFG.easyocr_languages), gpu=torch.cuda.is_available())
except Exception:
    easy_reader = easyocr.Reader(list(CFG.easyocr_languages), gpu=False)

arabic_spell = Corrector()
english_spell = SpellChecker(language='en')

write_health_snapshot({'stage': 'models_loaded', 'device': str(DEVICE)})
print(f'✅ اكتمل التحميل في {time.time() - load_start:.2f} ثانية')


In [ ]:
# ============================================================
# الخلية 8: تحليل feedback وبناء correction dictionary
# ============================================================
def normalize_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()


def build_correction_dict(min_frequency=None):
    min_frequency = min_frequency or CFG.correction_min_frequency
    if not FEEDBACK_CSV.exists():
        return {}
    df = pd.read_csv(FEEDBACK_CSV, encoding='utf-8-sig')
    if df.empty:
        return {}

    buckets = defaultdict(Counter)
    for _, row in df.iterrows():
        orig = normalize_text(row.get('original_text'))
        corr = normalize_text(row.get('corrected_text'))
        if orig and corr and orig != corr:
            buckets[orig][corr] += 1

    final_dict = {}
    for orig, cnt in buckets.items():
        best, freq = cnt.most_common(1)[0]
        if freq >= min_frequency:
            final_dict[orig] = best

    with open(CORRECTION_DICT_PATH, 'w', encoding='utf-8') as f:
        json.dump(final_dict, f, ensure_ascii=False, indent=2)
    log_event('correction_dict_built', {'entries': len(final_dict), 'min_frequency': min_frequency})
    return final_dict


def load_correction_dict():
    if CORRECTION_DICT_PATH.exists():
        with open(CORRECTION_DICT_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    return build_correction_dict()


def apply_correction_dict(text, correction_dict):
    if not text:
        return ''
    return ' '.join(correction_dict.get(tok, tok) for tok in text.split())


def feedback_summary():
    if not FEEDBACK_CSV.exists():
        print('⚠️ لا يوجد feedback بعد.')
        return None
    df = pd.read_csv(FEEDBACK_CSV, encoding='utf-8-sig')
    if df.empty:
        print('⚠️ feedback file فارغ.')
        return None
    print('إجمالي الصفوف:', len(df))
    display(df['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))
    display(df.head(10))
    return df

correction_dict = build_correction_dict()
print(f'✅ حجم قاموس التصحيح الحالي: {len(correction_dict)}')


In [ ]:
# ============================================================
# الخلية 9: دوال معالجة الصور قبل OCR
# ============================================================
def deskew_image(gray):
    coords = np.column_stack(np.where(gray < 250))
    if len(coords) < 50:
        return gray
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.3:
        return gray
    (h, w) = gray.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return rotated


def preprocess_image(img_bgr, use_clahe=True, use_denoise=True, adaptive_threshold=False, deskew=True):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if len(img_bgr.shape) == 3 else img_bgr.copy()
    if deskew:
        gray = deskew_image(gray)
    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)
    if use_denoise:
        gray = cv2.fastNlMeansDenoising(gray, h=20)
    if adaptive_threshold:
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 11)
    else:
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return gray, binary


def smart_word_segmentation(binary_img, min_box_width=None, min_box_height=None):
    min_box_width = min_box_width or CFG.min_box_width
    min_box_height = min_box_height or CFG.min_box_height
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 5))
    dilated = cv2.dilate(binary_img, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    word_boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > min_box_width and h > min_box_height:
            word_boxes.append((x, y, w, h))
    return sorted(word_boxes, key=lambda b: (b[1], b[0]))


def crop_safe(img, x, y, w, h):
    H, W = img.shape[:2]
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(W, x+w), min(H, y+h)
    return img[y1:y2, x1:x2]

print('✅ دوال preprocessing جاهزة')


In [ ]:
# ============================================================
# الخلية 10: OCR Ensemble + التصحيح
# ============================================================
def detect_language_safe(text):
    try:
        if not text or not str(text).strip():
            return 'unknown'
        return detect(str(text))
    except Exception:
        return 'unknown'


def correct_text(text):
    text = normalize_text(text)
    if not text:
        return ''
    lang = detect_language_safe(text)
    try:
        if lang == 'ar':
            return arabic_spell.contextual_correct(text)
        words = text.split()
        corrected = [english_spell.correction(w) or w for w in words]
        return ' '.join(corrected)
    except Exception:
        return text


def trocr_predict(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pixel_values = trocr_processor(images=Image.fromarray(rgb), return_tensors='pt').pixel_values.to(DEVICE)
    generated_ids = trocr_model.generate(pixel_values, max_length=64)
    text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return text


def tesseract_predict(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    cfg = '--oem 1 --psm 7'
    txt = pytesseract.image_to_string(rgb, lang='ara+eng', config=cfg)
    return txt.strip()


def recognize_word_ensemble(img_bgr, easyocr_raw=None):
    candidates = []
    if easyocr_raw is not None:
        _, text, conf = easyocr_raw
        candidates.append(('easyocr', normalize_text(text), float(conf)))
    try:
        txt = trocr_predict(img_bgr)
        if txt:
            candidates.append(('trocr', txt, 0.70))
    except Exception as e:
        log_event('trocr_failed', {'error': str(e)}, level='warning')
    try:
        txt = tesseract_predict(img_bgr)
        if txt:
            candidates.append(('tesseract', txt, 0.45))
    except Exception as e:
        log_event('tesseract_failed', {'error': str(e)}, level='warning')

    candidates = [c for c in candidates if c[1]]
    if not candidates:
        return '', 0.0, 'none', True, []
    best = max(candidates, key=lambda x: x[2])
    return best[1], float(best[2]), best[0], (best[2] < 0.5), candidates

print('✅ دوال OCR ensemble جاهزة')


In [ ]:
# ============================================================
# الخلية 11: تحميل المدخلات (PDF أو صورة)
# ============================================================
def load_pages_from_input(input_path, start_page=1, end_page=None, dpi=None):
    input_path = str(input_path)
    dpi = dpi or CFG.default_dpi
    ext = Path(input_path).suffix.lower()

    if ext == '.pdf':
        images = convert_from_path(input_path, dpi=dpi, first_page=start_page, last_page=end_page)
        pages = [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in images]
        page_numbers = list(range(start_page, start_page + len(pages)))
        return pages, page_numbers

    if ext in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}:
        img = cv2.imread(input_path)
        if img is None:
            raise ValueError(f'تعذر قراءة الصورة: {input_path}')
        return [img], [1]

    raise ValueError(f'نوع الملف غير مدعوم: {ext}')


In [ ]:
# ============================================================
# الخلية 12: دوال التلخيص السريع داخل قاعدة البيانات
# ============================================================
def get_status_counts():
    with get_conn() as conn:
        df = pd.read_sql_query('SELECT status, COUNT(*) AS cnt FROM handwriting_data GROUP BY status', conn)
    return {row['status']: int(row['cnt']) for _, row in df.iterrows()} if not df.empty else {}


def db_overview():
    counts = get_status_counts()
    print('📦 Status counts:')
    print(counts)
    with get_conn() as conn:
        recent = pd.read_sql_query('''
            SELECT image_id, predicted_text, raw_text, confidence, model_source, status, page_num
            FROM handwriting_data ORDER BY image_id DESC LIMIT 20
        ''', conn)
    display(recent)
    return counts, recent


In [ ]:
# ============================================================
# الخلية 13: دالة المعالجة الرئيسية
# ============================================================
def process_document(input_path=None, start_page=1, end_page=None, resume=True, adaptive_threshold=False,
                     delete_existing_range=False, show_preview=False):
    input_path = input_path or CFG.input_pdf_path
    run_id = datetime.now().strftime('run_%Y%m%d_%H%M%S')
    proc_start = time.time()
    correction_dict = load_correction_dict()

    checkpoint = load_checkpoint() if resume else None
    if checkpoint and checkpoint.get('input_path') == str(input_path):
        start_page = int(checkpoint.get('next_page', start_page))
        print(f'🔄 استئناف من الصفحة {start_page}')

    log_event('process_started', {'run_id': run_id, 'input_path': str(input_path), 'start_page': start_page, 'end_page': end_page})

    with get_conn() as conn:
        conn.execute('''
            INSERT OR REPLACE INTO processing_runs
            (run_id, started_at, input_path, start_page, end_page, dpi, words_processed, avg_confidence, status, notes)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (run_id, datetime.now().isoformat(), str(input_path), start_page, end_page or -1, CFG.default_dpi, 0, 0.0, 'running', ''))
        conn.commit()

    pages, page_numbers = load_pages_from_input(input_path, start_page=start_page, end_page=end_page, dpi=CFG.default_dpi)
    total_words = 0
    conf_acc = []

    with get_conn() as conn:
        if delete_existing_range and page_numbers:
            conn.execute('DELETE FROM handwriting_data WHERE page_num BETWEEN ? AND ?', (min(page_numbers), max(page_numbers)))
            conn.commit()
            log_event('existing_rows_deleted', {'start_page': min(page_numbers), 'end_page': max(page_numbers)})

        for page_img, actual_page in zip(pages, page_numbers):
            page_start = time.time()
            gray, binary = preprocess_image(page_img, adaptive_threshold=adaptive_threshold)

            try:
                detections = easy_reader.readtext(page_img, detail=1)
            except Exception as e:
                detections = []
                log_event('easyocr_page_failed', {'page': actual_page, 'error': str(e)}, level='warning')

            boxes_info = []
            if detections:
                for item in detections:
                    bbox, text, conf = item
                    pts = np.array(bbox, dtype=np.int32)
                    x, y, w, h = cv2.boundingRect(pts)
                    if w > CFG.min_box_width and h > CFG.min_box_height:
                        boxes_info.append(((x, y, w, h), item))
            else:
                for box in smart_word_segmentation(binary):
                    boxes_info.append((box, None))

            for (x, y, w, h), easy_item in boxes_info:
                crop = crop_safe(page_img, x, y, w, h)
                if crop.size == 0:
                    continue
                raw_text, conf, model_source, low_conf, candidates = recognize_word_ensemble(crop, easy_item)
                corrected = apply_correction_dict(correct_text(raw_text), correction_dict)
                _, buf = cv2.imencode('.png', crop)
                ts = datetime.now().isoformat()
                conn.execute('''
                    INSERT INTO handwriting_data (
                        image_data, predicted_text, raw_text, status, confidence, model_source,
                        page_num, x, y, w, h, created_at, updated_at, run_id
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''', (buf.tobytes(), corrected, raw_text, 'unverified', conf, model_source,
                      actual_page, x, y, w, h, ts, ts, run_id))
                total_words += 1
                conf_acc.append(conf)

            conn.commit()
            page_duration = time.time() - page_start
            save_checkpoint({
                'run_id': run_id,
                'input_path': str(input_path),
                'next_page': actual_page + 1,
                'processed_words': total_words,
                'timestamp': datetime.now().isoformat(),
            })
            log_event('page_processed', {
                'run_id': run_id,
                'page': actual_page,
                'boxes': len(boxes_info),
                'page_duration_sec': round(page_duration, 2),
                'total_words_so_far': total_words,
            })
            if show_preview:
                patches.cv2_imshow(cv2.resize(page_img, (0, 0), fx=0.35, fy=0.35))

    duration_sec = time.time() - proc_start
    avg_conf = float(np.mean(conf_acc)) if conf_acc else 0.0
    clear_checkpoint()

    stats = {
        'timestamp': datetime.now().isoformat(),
        'run_id': run_id,
        'input_path': str(input_path),
        'pages': len(page_numbers),
        'words': total_words,
        'avg_confidence': avg_conf,
        'duration_sec': duration_sec,
    }
    with open(PROCESSING_STATS_JSON, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    with get_conn() as conn:
        conn.execute('''
            UPDATE processing_runs
            SET ended_at=?, words_processed=?, avg_confidence=?, status=?, notes=?
            WHERE run_id=?
        ''', (datetime.now().isoformat(), total_words, avg_conf, 'completed', '', run_id))
        conn.commit()

    runs_df = pd.read_csv(RUNS_CSV, encoding='utf-8-sig') if RUNS_CSV.exists() else pd.DataFrame()
    runs_df = pd.concat([runs_df, pd.DataFrame([{
        'run_id': run_id,
        'timestamp': datetime.now().isoformat(),
        'input_path': str(input_path),
        'pages_processed': len(page_numbers),
        'words_processed': total_words,
        'verified_count': int(get_status_counts().get('verified', 0)),
        'avg_confidence': avg_conf,
        'status': 'completed',
        'duration_sec': duration_sec,
    }])], ignore_index=True)
    runs_df.to_csv(RUNS_CSV, index=False, encoding='utf-8-sig')

    write_health_snapshot({'stage': 'process_completed', 'run_id': run_id, 'duration_sec': duration_sec})
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print('✅ اكتملت المعالجة')
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    return stats


In [ ]:
# ============================================================
# الخلية 14: واجهة مراجعة الكلمات
# ============================================================
def launch_review_ui_v3(reviewer='user'):
    order_clause = 'confidence ASC, image_id ASC' if CFG.review_queue_order == 'low_confidence_first' else 'image_id ASC'
    with get_conn() as conn:
        df = pd.read_sql_query(f"SELECT * FROM handwriting_data WHERE status='unverified' ORDER BY {order_clause}", conn)
    if df.empty:
        print('✅ لا توجد كلمات للمراجعة.')
        return

    current = 0
    img = widgets.Image(format='png', width=360)
    text = widgets.Text(description='النص:', layout=widgets.Layout(width='95%'))
    raw_html = widgets.HTML()
    conf_bar = widgets.FloatProgress(min=0, max=1.0, description='الثقة:', layout={'width': '95%'})
    info = widgets.HTML()
    prog = widgets.IntProgress(min=0, max=max(0, len(df)-1), bar_style='info', layout={'width':'95%'})
    zoom_slider = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description='Zoom')

    def refresh():
        nonlocal current, df
        if df.empty:
            img.value = b''
            text.value = ''
            info.value = '<b>🏁 اكتملت المراجعة</b>'
            return
        current = max(0, min(current, len(df)-1))
        row = df.iloc[current]
        img.value = row['image_data']
        img.width = int(360 * zoom_slider.value)
        text.value = str(row['predicted_text'] or '')
        raw_html.value = f"<code>raw: {row['raw_text']}</code>"
        c = float(row['confidence'] or 0)
        conf_bar.value = c
        conf_bar.bar_style = 'danger' if c < 0.5 else ('warning' if c < 0.8 else 'success')
        info.value = f"<b>{current+1}/{len(df)}</b> | صفحة {row['page_num']} | المصدر: {row['model_source']} | image_id={row['image_id']}"
        prog.max = max(0, len(df)-1)
        prog.value = current

    def save_review(action):
        nonlocal current, df
        if df.empty:
            return
        row = df.iloc[current]
        rid = int(row['image_id'])
        original = normalize_text(row['predicted_text'])
        corrected = normalize_text(text.value)
        ts = datetime.now().isoformat()
        with get_conn() as conn:
            if action == 'confirm':
                conn.execute("UPDATE handwriting_data SET predicted_text=?, status='verified', updated_at=? WHERE image_id=?", (corrected, ts, rid))
                conn.execute("INSERT INTO review_events (timestamp, image_id, original_text, corrected_text, action, reviewer) VALUES (?, ?, ?, ?, ?, ?)", (ts, rid, original, corrected, 'confirm', reviewer))
            elif action == 'delete':
                conn.execute("DELETE FROM handwriting_data WHERE image_id=?", (rid,))
                conn.execute("INSERT INTO review_events (timestamp, image_id, original_text, corrected_text, action, reviewer) VALUES (?, ?, ?, ?, ?, ?)", (ts, rid, original, '', 'delete', reviewer))
            conn.commit()
        if action == 'confirm' and original != corrected:
            pd.DataFrame([{
                'timestamp': ts,
                'image_id': rid,
                'original_text': original,
                'corrected_text': corrected,
                'status': 'verified'
            }]).to_csv(FEEDBACK_CSV, mode='a', header=False, index=False, encoding='utf-8-sig')
        df = df.drop(df.index[current]).reset_index(drop=True)
        if current >= len(df) and len(df) > 0:
            current = len(df) - 1
        refresh()

    def go_prev(_):
        nonlocal current
        current = max(0, current-1)
        refresh()

    def go_next(_):
        nonlocal current
        if not df.empty:
            current = min(len(df)-1, current+1)
            refresh()

    btn_prev = widgets.Button(description='⬅ السابق', button_style='info')
    btn_confirm = widgets.Button(description='✅ تأكيد', button_style='success')
    btn_delete = widgets.Button(description='🗑 حذف', button_style='danger')
    btn_next = widgets.Button(description='التالي ➡', button_style='info')
    btn_prev.on_click(go_prev)
    btn_next.on_click(go_next)
    btn_confirm.on_click(lambda b: save_review('confirm'))
    btn_delete.on_click(lambda b: save_review('delete'))
    zoom_slider.observe(lambda change: refresh() if change['name']=='value' else None, names='value')

    display(widgets.VBox([
        prog, info, img, raw_html, text, zoom_slider,
        widgets.HBox([conf_bar]),
        widgets.HBox([btn_prev, btn_confirm, btn_delete, btn_next])
    ]))
    refresh()


In [ ]:
# ============================================================
# الخلية 15: إعادة بناء الجمل + واجهة مراجعة الجمل
# ============================================================
def reconstruct_sentences_df(y_tolerance=None, verified_only=False):
    y_tolerance = y_tolerance or CFG.line_y_tolerance
    query = "SELECT * FROM handwriting_data ORDER BY page_num, y, x"
    if verified_only:
        query = "SELECT * FROM handwriting_data WHERE status IN ('verified','sentence_corrected') ORDER BY page_num, y, x"
    with get_conn() as conn:
        words = pd.read_sql_query(query, conn)
    if words.empty:
        return pd.DataFrame()

    sentences = []
    for page in words['page_num'].unique():
        p_words = words[words['page_num'] == page].copy().sort_values(['y','x'])
        if p_words.empty:
            continue
        curr_line = [p_words.iloc[0].to_dict()]
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr_line[-1]['y']) <= y_tolerance:
                curr_line.append(row)
            else:
                sentences.append(curr_line)
                curr_line = [row]
        sentences.append(curr_line)

    out = []
    for line in sentences:
        preview = ' '.join(str(w.get('predicted_text') or '') for w in line).strip()
        lang = detect_language_safe(preview)
        sorted_line = sorted(line, key=lambda k: k['x'], reverse=(lang == 'ar'))
        sentence = ' '.join(str(w.get('predicted_text') or '') for w in sorted_line).strip()
        out.append({'page': line[0]['page_num'], 'y_anchor': line[0]['y'], 'lang': lang, 'text': sentence, 'word_ids': [w['image_id'] for w in sorted_line]})
    return pd.DataFrame(out)


def launch_sentence_review_ui_v2(y_tolerance=None):
    df_sent = reconstruct_sentences_df(y_tolerance=y_tolerance, verified_only=False)
    if df_sent.empty:
        print('⚠️ لا توجد بيانات كافية لمراجعة الجمل.')
        return

    current = 0
    info = widgets.HTML()
    text = widgets.Textarea(description='الجملة:', layout={'width':'95%','height':'100px'})
    prog = widgets.IntProgress(min=0, max=max(0, len(df_sent)-1), layout={'width':'95%'})

    def refresh():
        nonlocal current
        current = max(0, min(current, len(df_sent)-1))
        row = df_sent.iloc[current]
        text.value = row['text']
        info.value = f"<b>{current+1}/{len(df_sent)}</b> | صفحة {row['page']} | lang={row['lang']} | words={len(row['word_ids'])}"
        prog.value = current

    def save_current(_):
        nonlocal current
        row = df_sent.iloc[current]
        corrected = normalize_text(text.value)
        original = normalize_text(row['text'])
        ts = datetime.now().isoformat()
        with get_conn() as conn:
            for wid in row['word_ids']:
                conn.execute("UPDATE handwriting_data SET status='sentence_corrected', updated_at=? WHERE image_id=?", (ts, int(wid)))
            conn.commit()
        pd.DataFrame([{
            'timestamp': ts,
            'image_id': None,
            'original_text': original,
            'corrected_text': corrected,
            'status': f"sent_rev_p{row['page']}_y{row['y_anchor']}"
        }]).to_csv(FEEDBACK_CSV, mode='a', header=False, index=False, encoding='utf-8-sig')
        orig_words = original.split()
        corr_words = corrected.split()
        if len(orig_words) == len(corr_words):
            derived = []
            for o, c in zip(orig_words, corr_words):
                if o != c:
                    derived.append({'timestamp': ts, 'image_id': None, 'original_text': o, 'corrected_text': c, 'status': 'sentence_derived'})
            if derived:
                pd.DataFrame(derived).to_csv(FEEDBACK_CSV, mode='a', header=False, index=False, encoding='utf-8-sig')
        log_event('sentence_review_saved', {'page': int(row['page']), 'word_count': len(row['word_ids'])})
        if current < len(df_sent)-1:
            current += 1
        refresh()

    def go_prev(_):
        nonlocal current
        current = max(0, current-1)
        refresh()

    def go_next(_):
        nonlocal current
        current = min(len(df_sent)-1, current+1)
        refresh()

    btn_prev = widgets.Button(description='⬅ السابق', button_style='info')
    btn_save = widgets.Button(description='✅ حفظ وتأكيد', button_style='success')
    btn_next = widgets.Button(description='التالي ➡', button_style='info')
    btn_prev.on_click(go_prev)
    btn_next.on_click(go_next)
    btn_save.on_click(save_current)

    display(widgets.VBox([prog, info, text, widgets.HBox([btn_prev, btn_save, btn_next])]))
    refresh()


In [ ]:
# ============================================================
# الخلية 16: التصدير والتدريب
# ============================================================
def export_finetuning_dataset(output_dir=None, val_ratio=0.1):
    output_dir = Path(output_dir or (DIRS['exports'] / 'hf_training_dataset'))
    img_dir = output_dir / 'images'
    img_dir.mkdir(parents=True, exist_ok=True)

    with get_conn() as conn:
        df_verified = pd.read_sql_query('''
            SELECT image_id, image_data, predicted_text, status
            FROM handwriting_data
            WHERE status IN ('verified', 'sentence_corrected')
            ORDER BY image_id
        ''', conn)
    if df_verified.empty:
        print('⚠️ لا توجد بيانات موثقة بعد.')
        return None

    records = []
    for _, row in df_verified.iterrows():
        fname = f"img_{row['image_id']}.png"
        with open(img_dir / fname, 'wb') as f:
            f.write(row['image_data'])
        records.append({'image': fname, 'text': normalize_text(row['predicted_text']), 'verified': True})

    np.random.shuffle(records)
    split_idx = int(len(records) * (1 - val_ratio))
    train_data = records[:split_idx]
    val_data = records[split_idx:]

    def save_jsonl(data, path):
        with open(path, 'w', encoding='utf-8') as f:
            for rec in data:
                f.write(json.dumps(rec, ensure_ascii=False) + '
')

    save_jsonl(train_data, output_dir / 'train.jsonl')
    save_jsonl(val_data, output_dir / 'val.jsonl')
    print(f'✅ تم التصدير إلى: {output_dir}')
    print(f'عدد العينات: {len(records)} | train={len(train_data)} | val={len(val_data)}')
    return output_dir


def push_to_huggingface(hf_repo_id, hf_token=None, local_dataset_dir=None):
    from huggingface_hub import HfApi, login
    hf_token = hf_token or HF_TOKEN
    local_dataset_dir = str(local_dataset_dir or (DIRS['exports'] / 'hf_training_dataset'))
    if not hf_token:
        print('⚠️ لا يوجد HF_TOKEN')
        return
    if not os.path.exists(local_dataset_dir):
        print('⚠️ صدّر البيانات أولًا')
        return
    login(token=hf_token)
    api = HfApi()
    api.create_repo(repo_id=hf_repo_id, repo_type='dataset', exist_ok=True)
    api.upload_folder(folder_path=local_dataset_dir, repo_id=hf_repo_id, repo_type='dataset', commit_message=f"Update dataset {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f'✅ تم الرفع: https://huggingface.co/datasets/{hf_repo_id}')


def finetune_trocr_lora(min_samples=100, epochs=3, batch_size=4, lr=5e-5):
    from peft import get_peft_model, LoraConfig, TaskType
    from torch.optim import AdamW
    from torch.utils.data import Dataset as TorchDataset, DataLoader
    global trocr_model

    with get_conn() as conn:
        df_v = pd.read_sql_query("SELECT image_data, predicted_text FROM handwriting_data WHERE status IN ('verified', 'sentence_corrected')", conn)
    if len(df_v) < min_samples:
        print(f'⚠️ لديك {len(df_v)} عينة فقط. الحد الأدنى الحالي = {min_samples}')
        return

    class HandwritingDataset(TorchDataset):
        def __init__(self, df):
            self.df = df.reset_index(drop=True)
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            pixel_values = trocr_processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
            labels = trocr_processor.tokenizer(normalize_text(row['predicted_text']), return_tensors='pt', padding='max_length', truncation=True, max_length=64).input_ids.squeeze(0)
            labels[labels == trocr_processor.tokenizer.pad_token_id] = -100
            return {'pixel_values': pixel_values, 'labels': labels}

    lora_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, target_modules=['query', 'value'], lora_dropout=0.1)
    model = get_peft_model(trocr_model, lora_config)
    model.train()
    loader = DataLoader(HandwritingDataset(df_v), batch_size=batch_size, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=lr)

    log_event('finetune_started', {'samples': len(df_v), 'epochs': epochs, 'batch_size': batch_size})
    for epoch in range(epochs):
        total_loss = 0.0
        for batch in loader:
            out = model(pixel_values=batch['pixel_values'].to(DEVICE), labels=batch['labels'].to(DEVICE))
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += float(out.loss.item())
        print(f'Epoch {epoch+1}/{epochs} | Loss={total_loss/max(1, len(loader)):.4f}')

    lora_path = DIRS['cache'] / 'trocr_lora_finetuned'
    model.save_pretrained(str(lora_path))
    trocr_processor.save_pretrained(str(lora_path))
    trocr_model = model
    log_event('finetune_completed', {'save_path': str(lora_path)})
    print('✅ تم حفظ LoRA في:', lora_path)


In [ ]:
# ============================================================
# الخلية 17: استخراج قاموس ثنائي اللغة + نسخة احتياطية
# ============================================================
def extract_bilingual_vocab(output_path=None):
    output_path = output_path or (DIRS['exports'] / 'bilingual_vocab.csv')
    with get_conn() as conn:
        words = pd.read_sql_query('''
            SELECT DISTINCT predicted_text, page_num, x, y, status
            FROM handwriting_data
            WHERE status IN ('verified', 'sentence_corrected')
            ORDER BY page_num, y, x
        ''', conn)
    if words.empty:
        print('⚠️ لا توجد بيانات موثقة.')
        return None

    vocab_pairs = []
    for page in words['page_num'].unique():
        p_words = words[words['page_num'] == page].copy()
        curr_line = [p_words.iloc[0].to_dict()]
        lines = []
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr_line[-1]['y']) <= CFG.line_y_tolerance:
                curr_line.append(row)
            else:
                lines.append(curr_line)
                curr_line = [row]
        lines.append(curr_line)
        for line in lines:
            texts = [normalize_text(w['predicted_text']) for w in line if normalize_text(w['predicted_text'])]
            en_words = [t for t in texts if all(ord(c) < 128 for c in t.replace(' ', ''))]
            ar_words = [t for t in texts if any('؀' <= c <= 'ۿ' for c in t)]
            if en_words or ar_words:
                vocab_pairs.append({'english': ' | '.join(en_words), 'arabic': ' | '.join(ar_words), 'page': page})

    df = pd.DataFrame(vocab_pairs)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'✅ تم استخراج {len(df)} صفًا إلى {output_path}')
    display(df.head(20))
    return df


def create_project_backup(label=None):
    label = label or datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_dir = DIRS['backups'] / f'backup_{label}'
    backup_dir.mkdir(parents=True, exist_ok=True)
    for path in [DB_PATH, FEEDBACK_CSV, PROCESSING_STATS_JSON, APP_LOG, EVENTS_JSONL, CORRECTION_DICT_PATH]:
        if Path(path).exists():
            shutil.copy2(path, backup_dir / Path(path).name)
    print('✅ تم إنشاء نسخة احتياطية في:', backup_dir)
    return backup_dir


In [ ]:
# ============================================================
# الخلية 18: Monitoring Dashboard
# ============================================================
def read_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def tail_text_file(path, n=20):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    return [line.rstrip('
') for line in lines[-n:]]


def monitoring_dashboard(show_plots=True):
    print('📍 Monitoring Dashboard')
    print('='*70)
    stats = read_json_if_exists(PROCESSING_STATS_JSON)
    health = read_json_if_exists(HEALTH_JSON)
    bootstrap = read_json_if_exists(NOTEBOOK_BOOTSTRAP_SUMMARY)

    if bootstrap:
        print('
🧩 Bootstrap summary from attached materials')
        print(json.dumps(bootstrap, ensure_ascii=False, indent=2))
    if stats:
        print('
📊 Latest processing stats')
        print(json.dumps(stats, ensure_ascii=False, indent=2))
    if health:
        print('
🖥️ Latest health snapshot')
        print(json.dumps(health, ensure_ascii=False, indent=2))

    counts = get_status_counts()
    print('
🗂️ DB status counts:', counts)

    if FEEDBACK_CSV.exists():
        df_fb = pd.read_csv(FEEDBACK_CSV, encoding='utf-8-sig')
        print(f'
📝 Feedback rows: {len(df_fb)}')
        if not df_fb.empty:
            display(df_fb.tail(10))
            display(df_fb['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))

    if RUNS_CSV.exists():
        runs = pd.read_csv(RUNS_CSV, encoding='utf-8-sig')
        if not runs.empty:
            print('
🏃 آخر التشغيلات')
            display(runs.tail(10))
            if show_plots:
                vals = pd.to_numeric(runs['words_processed'], errors='coerce').fillna(0).values
                plt.figure(figsize=(10,4))
                plt.plot(vals, marker='o')
                plt.title('Words processed per run')
                plt.xlabel('Run index')
                plt.ylabel('Words')
                plt.grid(True)
                plt.show()

    print('
📄 tail(app.log)')
    for line in tail_text_file(APP_LOG, n=20):
        print(line)

    print('
📄 tail(events.jsonl)')
    for line in tail_text_file(EVENTS_JSONL, n=20):
        print(line)


In [ ]:
# ============================================================
# الخلية 19: أوامر تشغيل سريعة
# ============================================================
print('جاهز ✅')
print('أمثلة التشغيل:')
print('1) process_document(input_path=CFG.input_pdf_path, start_page=1, end_page=3, resume=True)')
print('2) launch_review_ui_v3()')
print('3) launch_sentence_review_ui_v2()')
print('4) feedback_summary()')
print('5) monitoring_dashboard()')
print('6) export_finetuning_dataset()')
print('7) create_project_backup()')


## ترتيب الاستخدام المقترح

1. شغّل الخلايا 1 → 8 مرة واحدة.
2. لو عندك feedback أو logs قديمة، ارفعها من الخلية 5 وادمجها.
3. شغّل `process_document(...)`.
4. راجع الكلمات من `launch_review_ui_v3()`.
5. راجع الجمل من `launch_sentence_review_ui_v2()`.
6. راقب التقدم من `monitoring_dashboard()`.
7. عند توفر بيانات كافية:
   - `export_finetuning_dataset()`
   - `finetune_trocr_lora()`
8. خُد نسخة احتياطية من `create_project_backup()`.

## ملفات المتابعة التي سينتجها المشروع
- `ocr_runtime.log`
- `ocr_events.jsonl`
- `processing_stats.json`
- `runs_history.csv`
- `system_health.json`
- `user_corrections_feedback.csv`
- `ocr_checkpoint.json`

## ملاحظات تشغيل
- لو Colab وقع، checkpoint هيساعدك تكمل من آخر صفحة تقريبًا.
- لو الاستهلاك عالي، عالج الصفحات على دفعات صغيرة.
- دمج feedback القديم بدري بيحسن التصحيح التلقائي من البداية.
